In [1]:
# LOAD
from sentence_transformers import SentenceTransformer
from sentence_transformers import CrossEncoder
from rouge_score import rouge_scorer
from bert_score import BERTScorer
import faiss
import numpy as np
import re


chunks = np.load("Data/chunks.npy", allow_pickle=True)
index = faiss.read_index("Data/index.faiss")

embedding_model = SentenceTransformer("intfloat/multilingual-e5-small")
reranker_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')


# BERTScore and ROUGE-L metric models
class _RomanianTokenizer:
    """Keeps Romanian diacritics that rouge_score's default tokenizer would drop."""
    def tokenize(self, text):
        return re.findall(r"\w+", text.lower(), flags=re.UNICODE)

_rouge_scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False, tokenizer=_RomanianTokenizer())

_bert_scorer = BERTScorer(lang="ro", rescale_with_baseline=False,)

c:\Users\phili\Personal\University\Licenta\Proiect\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11002.14it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 11840.35it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when 

In [2]:
# RETRIEVAL 
# retrieve top 20 chunks with the embedding model -> choose top 5 with the reranker model

def retrieve(query, k_initial=20, k_final=5):
    # Pasul A: Retrieval inițial (extrage mai multe fragmente)
    query_prefixed = "query: " + query
    q_emb = embedding_model.encode([query_prefixed])
    distances, indices = index.search(np.array(q_emb), k_initial)
    
    initial_chunks = [chunks[i] for i in indices[0]]

    if k_final == 0:  # 0 means skip reranking
        return initial_chunks


    # Pasul B: Reranking (Cross-Encoding)

    # Modelul compară perechi (întrebare, fragment) și dă un scor de relevanță
    sentence_combinations = [[query, chunk] for chunk in initial_chunks]
    rerank_scores = reranker_model.predict(sentence_combinations)
    
    # Pasul C: Sortăm fragmentele în funcție de scorul Reranker-ului
    reranked_results = sorted(
        list(zip(initial_chunks, rerank_scores)), 
        key=lambda x: x[1], 
        reverse=True
    )
    
    # Returnăm doar primele k_final cele mai bune
    return [res[0] for res in reranked_results[:k_final]]

QUESTION ANSWERING LLM

In [3]:
# QUESTION ANSWERING LLM
import requests

def ask_llm(question, context_chunks):
    context = "\n\n--- FRAGMENT NOU ---\n\n".join(context_chunks)

    prompt = f"""
Ești un asistent virtual oficial care ajută candidații cu informații despre admiterea la FMI UBB Cluj.
Trebuie să fii clar, concis și să răspunzi DOAR pe baza contextului oferit.

REGULI STRICTE:
1. Răspunde EXCLUSIV în limba română.
2. Folosește DOAR informațiile din secțiunea "Context". Nu folosi cunoștințele tale generale.
3. Dacă răspunsul nu se află în "Context", nu încerca să ghicești. Scrie EXACT următoarea propoziție: "Nu dețin această informație conform datelor extrase de pe site-ul oficial."
4. Nu inventa formule de calcul, date sau taxe.

Context:
{context}

Întrebare:
{question}

Răspuns:
"""

    response = requests.post(
        "http://localhost:1234/v1/chat/completions",
        json={
            "model": "google/gemma-4-e4b",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.1
        }
    )


    data = response.json()
    
    if "choices" not in data:
        print("Eroare primită de la LM Studio:", data)
        return "Eroare la generarea răspunsului."

    return data["choices"][0]["message"]["content"]

SCORING LLM

In [4]:
# SCORING SYSTEM LLM
def score_llm(question, context_chunks, answer, gold_answer):
    context = "\n\n--- FRAGMENT NOU ---\n\n".join(context_chunks)

    prompt = f"""
You are a ruthless and objective evaluator of a RAG (Retrieval-Augmented Generation) chatbot for university admissions.
Your task is to evaluate the chatbot's answer. The chatbot answered on a context it has been given.
You will receive the question, the chatbot's answer for it, the context based on which the chatbot has answered, and a gold answer.
Use the reference context to determine whether the chatbot hallucinated or not.
Use the gold answer to determine the factual correctness and the completeness of the chatbot's answer.

When the Golden Answer contains facts absent from the Reference Context, this
indicates a RETRIEVAL failure, not a hallucination — score no_hallucination based
strictly on consistency with the context, and score factual/completeness based on
the Golden Answer.

Question:
{question}

Chatbot Answer:
{answer}

Reference Context:
{context}

Gold Answer:
{gold_answer}

---
SCORING RUBRIC (Be extremely strict!):
1. Factual correctness (0-10): 
   - 10 = Perfectly aligned with the golden answer.
   - 5 = Partially correct, but contains errors.
   - 0 = The information is completely wrong or invented.
2. Completeness (0-10): Did the answer cover all the key facts present in the Golden Answer?
   - 10 = All key facts from the gold answer are present.
   - 5 = Some key facts missing.
   - 0 = Most key facts missing.
3. Clarity (0-10): Is the answer easy to read and in Romanian? (Score 0 if the answer is in English!).
4. Hallucinations penalty (0-10): 
   - 10 = NO hallucinations at all. Only used the context.
   - 0 = The model invented rules, formulas, or deadlines NOT present in the Reference Context.

Return ONLY valid JSON format:
{{
  "factual": <int>,
  "complete": <int>,
  "clarity": <int>,
  "no_hallucination": <int>,
  "brief_reason": "<1-2 sentences explaining exactly why points were deducted>"
}}
"""

    response = requests.post(
        "http://localhost:1234/v1/chat/completions",
        json={
            "model": "google/gemma-2-9b",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0
        }
    )


    data = response.json()
    
    if "choices" not in data:
        print("Eroare primită de la LM Studio:", data)
        return "Eroare la generarea răspunsului."

    return data["choices"][0]["message"]["content"]

In [5]:
# QA Model function
def answer_question(question, relevant_chunks):
    answer = ask_llm(question, relevant_chunks)
    return answer

# LLM-as-a-Judge Model function
def score_answer(question, relevant_chunks, answer, gold_answer):
    answer = score_llm(question, relevant_chunks, answer, gold_answer)
    return answer

Model answers question & Judge scores them

In [14]:
import time
import json
import pandas as pd



def load_questions_and_gold_answers(questionSet):
    if questionSet == "allQuestions":
        with open("Data/Gold Answers/answers.json", "r", encoding="utf-8") as f:
            return json.load(f)
    elif questionSet == "100questionsDataset":
        with open("Automated Experiments/100questionsDataset.json", "r", encoding="utf-8") as f:
            return json.load(f)


def save_results(experimentType, path, answers, scores, judgeAnswers=True):
    df = pd.DataFrame(answers)
    df.to_csv(experimentType + "/" + path + "/answers.csv", index=False, encoding="utf-8")

    if judgeAnswers:
        df = pd.DataFrame(scores)
        df.to_csv(experimentType + "/" + path + "/scores.csv", index=False, encoding="utf-8")


# Compute the BERTScore and ROUGE-L metrics of comparing the chatbot answer to the gold answer.
# Returns (bertscore_f1, rougeL_f1).
def compute_metrics(chatbotAnswer, goldAnswer):
    if not chatbotAnswer or not goldAnswer:
        return 0.0, 0.0

    rougeL = _rouge_scorer.score(goldAnswer, chatbotAnswer)["rougeL"].fmeasure

    _, _, F1 = _bert_scorer.score([chatbotAnswer], [goldAnswer])
    bertscore = F1[0].item()

    return round(bertscore, 4), round(rougeL, 4)



# Compute the total score of the judge given scores.
# Returns the weighted sum of factual, complete, clarity and no_hallucination scores.
def compute_score(raw: str) -> dict:
    WEIGHTS = {
        "factual": 0.40,
        "complete": 0.25,
        "clarity": 0.15,
        "no_hallucination": 0.20,
    }

    raw = raw.strip()
    start, end = raw.find("{"), raw.rfind("}")
    if start != -1 and end > start:
        data = json.loads(raw[start:end + 1])
    else:
        raise ValueError(f"Nu am putut extrage JSON din răspunsul judge-ului:\n{raw}")

    components = {}
    for key in WEIGHTS:
        val = float(data.get(key, 0))          
        components[key] = max(0.0, min(10.0, val))  

    final = sum(WEIGHTS[k] * components[k] for k in WEIGHTS)
    data["final_score"] = round(final, 2)
    return data


# path: The folder where the csv files will be
# experimentType: Still part of the whole file path, determines the experiment category
# initialK & finalK: For the retrieval function
# questionSet: answer a specified set of questions (enters the if-branch)
# question: ask one specific question (enters the else-branch)
# metrics: Whether we want BERTScore and ROUGE-L metrics to be computed and appended to the csv files
# judgeAnswers: if False, the LLM-as-a-Judge will no longer be activated
def answerAndScore(
   path=None, initialK=20, finalK=5, question=None, questionSet="allQuestions", 
   experimentType="Manual Experiments", metrics=False, judgeAnswers=True
):
    questions_and_gold_answers = load_questions_and_gold_answers(questionSet)
    answers = []
    scores = []
    totalTime = 0
    
    if path != None and question == None:
        for QA in questions_and_gold_answers:
            try:
                start = time.time()
                relevant_chunks = retrieve(QA["question"], initialK, finalK)

                chatbotAnswer = answer_question(QA["question"], relevant_chunks)
                if chatbotAnswer == "Eroare la generarea răspunsului.":
                    raise Exception("Chatbot answer generation failed")
                print("Answered question ", QA["id"]) 

                current_answer = {
                    "id": QA["id"],
                    "question": QA["question"],
                    "category": QA["category"],
                    "answer": chatbotAnswer,
                    "gold answer": QA["answer"]
                }
                if metrics:
                    bertscore, rougeL = compute_metrics(chatbotAnswer, QA["answer"])
                    current_answer["bertscore"] = bertscore
                    current_answer["rougeL"] = rougeL

                answers.append(current_answer)

                if not judgeAnswers:  # skip Judge section
                    elapsed = int(time.time() - start) ; minutes = elapsed // 60 ; seconds = elapsed % 60
                    print(f"Time: {minutes}m{seconds}s")
                    totalTime += elapsed
                    save_results(experimentType, path, answers, -1, judgeAnswers)
                    continue

                raw = score_answer(QA["question"], relevant_chunks, chatbotAnswer, QA["answer"])
                if raw == "Eroare la generarea răspunsului.":
                    raise Exception("Chatbot answer generation failed")
                try:
                    score = compute_score(raw)
                except (ValueError, KeyError, json.JSONDecodeError) as e:
                    score = {"error": str(e), "raw": raw, "final_score": None}

                print("Scored answer for question", QA["id"])
                scores.append({
                    "id": QA["id"],
                    "question": QA["question"],
                    "category": QA["category"],
                    "answer": chatbotAnswer,
                    "score": json.dumps(score, ensure_ascii=False)
                })

                elapsed = int(time.time() - start) ; minutes = elapsed // 60 ; seconds = elapsed % 60
                print(f"Time: {minutes}m{seconds}s")
                totalTime += elapsed

                save_results(experimentType, path, answers, scores)
            
            except Exception as e:
                print(f"Question {QA["id"]} failed: {e}")
                elapsed = int(time.time() - start) ; minutes = elapsed // 60 ; seconds = elapsed % 60
                print(f"Time: {minutes}m{seconds}s")
                totalTime += elapsed
                continue
            
        print(f"Total time: {totalTime//60}m{totalTime%60}s")


    elif path == None and question != None:
        relevant_chunks = retrieve(question, initialK, finalK)
        answer = answer_question(question, relevant_chunks)
        print(answer) 
        score = score_answer(question, relevant_chunks, answer)
        print("\n\n",score)
    else:
        print("Wrong parameter input")

Analyse scores

In [7]:
import pandas as pd
import json


def parse_score(text):
    text = str(text).strip()

    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()

    return json.loads(text)


def analyse_scores(path, file="scores.csv", experimentType="Manual Experiments"):
    df = pd.read_csv(experimentType + "/" + path + "/" + file)

    df["score_dict"] = df["score"].apply(parse_score)

    df["final_score"] = df["score_dict"].apply(lambda x: x.get("final_score", 0))
    df["factual"] = df["score_dict"].apply(lambda x: x.get("factual", 0))
    df["complete"] = df["score_dict"].apply(lambda x: x.get("complete", 0))
    df["clarity"] = df["score_dict"].apply(lambda x: x.get("clarity", 0))
    df["no_hallucination"] = df["score_dict"].apply(lambda x: x.get("no_hallucination", 0))

    print("Average final:", df["final_score"].mean())
    print("Average factual:", df["factual"].mean())
    print("Average complete:", df["complete"].mean())
    print("Average clarity:", df["clarity"].mean())
    print("Average non-hallucinative:", df["no_hallucination"].mean())
    print("-------------------------------------")
    print(df.groupby("category")["final_score"].mean())

Manual Experiments

In [3]:
#answerAndScore("1_newInput")
# Total time: 5m9s

#analyse_scores("1_newInput")

In [2]:
#answerAndScore("2_changedModels")
# Total time: 

# Improvement is very clear: the QA model answers much more reliably.

#analyse_scores("2_changedModels")

In [1]:
#answerAndScore(path="trackChanges", questionSet="100questionsDataset", metrics=True, judgeAnswers=False)

#analyse_scores("trackChanges")

### RUN EXPERIMENTS

In [ ]:
# TOTAL AMOUNT OF QUESTIONS ANSWERED: 800  (8 iterations * 100 questions)

# EXPERIMENT 1: WITH VS. WITHOUT RERANKING
# We will test the chatbot's performance without reranking (just top 5) and with reranking (top 20, reranked 5)
def experiment1():
    print("-----EXPERIMENT 1-----")

    answerAndScore(path="Experiment 1/WithoutReranking", questionSet="100questionsDataset", initialK=5, finalK=0, experimentType="Automated Experiments", metrics=True, judgeAnswers=False)
    print("--Finished: Without reranking--\n")

    answerAndScore(path="Experiment 1/WithReranking", questionSet="100questionsDataset", initialK=20, finalK=5, experimentType="Automated Experiments", metrics=True, judgeAnswers=False)
    print("--Finished: With reranking--\n")


# EXPERIMENT 2: DIFFERENT CHUNK SIZES
# We will test the chatbot's performance on chunk size 500, 1000 and 1500, all with overlap 200.
import chunkingAndEmbeddings

def experiment2():
    print("\n-----EXPERIMENT 2-----")

    chunkingAndEmbeddings.computeChunksAndEmbeddings(500, 200, embedding_model)
    answerAndScore(path="Experiment 2/ChunkSize500", questionSet="100questionsDataset", initialK=20, finalK=5, experimentType="Automated Experiments", metrics=True, judgeAnswers=False)
    print("--Finished: 500 chunks--\n")
    
    chunkingAndEmbeddings.computeChunksAndEmbeddings(1000, 200, embedding_model)
    answerAndScore(path="Experiment 2/ChunkSize1000", questionSet="100questionsDataset", initialK=20, finalK=5, experimentType="Automated Experiments", metrics=True, judgeAnswers=False)
    print("--Finished: 1000 chunks--\n")
    
    chunkingAndEmbeddings.computeChunksAndEmbeddings(1500, 200, embedding_model)
    answerAndScore(path="Experiment 2/ChunkSize1500", questionSet="100questionsDataset", initialK=20, finalK=5, experimentType="Automated Experiments", metrics=True, judgeAnswers=False)
    print("--Finished: 1500 chunks--\n")


# EXPERIMENT 3: DIFFERENT TOP-K VALUES
# We will test the chatbot's performance on different top-k values.
#  = out of the top 20, we will take reranked top 3, 5 and 8
def experiment3():
    print("\n-----EXPERIMENT 3-----")

    answerAndScore(path="Experiment 3/Top3", questionSet="100questionsDataset", initialK=20, finalK=3, experimentType="Automated Experiments", metrics=True, judgeAnswers=False)
    print("--Finished: top 3--\n")
    
    answerAndScore(path="Experiment 3/Top5", questionSet="100questionsDataset", initialK=20, finalK=5, experimentType="Automated Experiments", metrics=True, judgeAnswers=False)
    print("--Finished: top 5--\n")
    
    answerAndScore(path="Experiment 3/Top8", questionSet="100questionsDataset", initialK=20, finalK=8, experimentType="Automated Experiments", metrics=True, judgeAnswers=False)
    print("--Finished: top 8--\n")


def reset_after_experiment():
    chunkingAndEmbeddings.computeChunksAndEmbeddings(1000, 200)


def initiateExperiments():
    try:
        experiment1()
    except Exception as e:
        print("Experiment 1 failed: ", e)
    #reset_after_experiment()
    
    try:
        experiment2()
    except Exception as e:
        print("Experiment 2 failed: ", e)
    #reset_after_experiment()

    try:
        experiment3()
    except Exception as e:
        print("Experiment 3 failed: ", e)
    #reset_after_experiment()

    print("\n\n\nProcess finished!")

In [ ]:
#initiateExperiments()


-----EXPERIMENT 3-----
Answered question  1
Time: 0m14s
Answered question  3
Time: 0m11s
Answered question  7
Time: 0m11s
Answered question  22
Time: 0m16s
Answered question  23
Time: 0m12s
Answered question  30
Time: 0m18s
Answered question  43
Time: 0m14s
Answered question  47
Time: 0m21s
Answered question  50
Time: 0m14s
Answered question  55
Time: 0m14s
Answered question  57
Time: 0m15s
Answered question  69
Time: 0m17s
Answered question  70
Time: 0m13s
Answered question  73
Time: 0m19s
Answered question  86
Time: 0m16s
Answered question  97
Time: 0m15s
Answered question  99
Time: 0m13s
Answered question  101
Time: 0m12s
Answered question  108
Time: 0m14s
Answered question  111
Time: 0m20s
Answered question  120
Time: 0m15s
Answered question  123
Time: 0m13s
Answered question  130
Time: 0m15s
Answered question  137
Time: 0m14s
Answered question  140
Time: 0m17s
Answered question  152
Time: 0m13s
Answered question  161
Time: 0m12s
Answered question  165
Time: 0m11s
Answered questio

### ANALYSE EXPERIMENT RESULTS

In [15]:
import pandas as pd
import json


def compareExperiment1Results():
    print("EXPERIMENT 1")

    df_withoutReranking = pd.read_csv("Automated Experiments/Experiment 1/WithoutReranking/answers.csv")
    df_withReranking = pd.read_csv("Automated Experiments/Experiment 1/WithReranking/answers.csv")
    
    avg_bertscore_withoutReranking = round(df_withoutReranking["bertscore"].mean(),4)
    avg_bertscore_withReranking = round(df_withReranking["bertscore"].mean(),4)

    avg_rougeL_withoutReranking = round(df_withoutReranking["rougeL"].mean(),4)
    avg_rougeL_withReranking = round(df_withReranking["rougeL"].mean(),4)

    print("Average BERTScore:\n", 
    f"Without reranking: {avg_bertscore_withoutReranking}\n",
    f"With reranking: {avg_bertscore_withReranking}")
    print("Average ROUGE-L:\n", 
    f"Without reranking: {avg_rougeL_withoutReranking}\n",
    f"With reranking: {avg_rougeL_withReranking}\n")


def compareExperiment2Results():
    print("EXPERIMENT 2")

    df_500chunks = pd.read_csv("Automated Experiments/Experiment 2/ChunkSize500/answers.csv")
    df_1000chunks = pd.read_csv("Automated Experiments/Experiment 2/ChunkSize1000/answers.csv")
    df_1500chunks = pd.read_csv("Automated Experiments/Experiment 2/ChunkSize1500/answers.csv")
    
    avg_bertscore_500chunks = round(df_500chunks["bertscore"].mean(),4)
    avg_bertscore_1000chunks = round(df_1000chunks["bertscore"].mean(),4)
    avg_bertscore_1500chunks = round(df_1500chunks["bertscore"].mean(),4)

    avg_rougeL_500chunks = round(df_500chunks["rougeL"].mean(),4)
    avg_rougeL_1000chunks = round(df_1000chunks["rougeL"].mean(),4)
    avg_rougeL_1500chunks = round(df_1500chunks["rougeL"].mean(),4)

    print("Average BERTScore:\n", 
    f"500 chunks: {avg_bertscore_500chunks}\n",
    f"1000 chunks: {avg_bertscore_1000chunks}\n",
    f"1500 chunks: {avg_bertscore_1500chunks}")
    print("Average ROUGE-L:\n", 
    f"500 chunks: {avg_rougeL_500chunks}\n",
    f"1000 chunks: {avg_rougeL_1000chunks}\n",
    f"1500 chunks: {avg_rougeL_1500chunks}\n")


def compareExperiment3Results():
    print("EXPERIMENT 3")

    df_top3 = pd.read_csv("Automated Experiments/Experiment 3/Top3/answers.csv")
    df_top5 = pd.read_csv("Automated Experiments/Experiment 3/Top5/answers.csv")
    df_top8 = pd.read_csv("Automated Experiments/Experiment 3/Top8/answers.csv")
    
    avg_bertscore_top3 = round(df_top3["bertscore"].mean(),4)
    avg_bertscore_top5 = round(df_top5["bertscore"].mean(),4)
    avg_bertscore_top8 = round(df_top8["bertscore"].mean(),4)

    avg_rougeL_top3 = round(df_top3["rougeL"].mean(),4)
    avg_rougeL_top5 = round(df_top5["rougeL"].mean(),4)
    avg_rougeL_top8 = round(df_top8["rougeL"].mean(),4)

    print("Average BERTScore:\n", 
    f"Top 3: {avg_bertscore_top3}\n",
    f"Top 5: {avg_bertscore_top5}\n",
    f"Top 8: {avg_bertscore_top8}")
    print("Average ROUGE-L:\n", 
    f"Top 3: {avg_rougeL_top3}\n",
    f"Top 5: {avg_rougeL_top5}\n",
    f"Top 8: {avg_rougeL_top8}\n")


def listAllExperimentResults():
    compareExperiment1Results()
    compareExperiment2Results()
    compareExperiment3Results()


In [ ]:
listAllExperimentResults()

EXPERIMENT 1
Average BERTScore:
 Without reranking: 0.8117
 With reranking: 0.8017
Average ROUGE-L:
 Without reranking: 0.5039
 With reranking: 0.4743

EXPERIMENT 2
Average BERTScore:
 500 chunks: 0.8085
 1000 chunks: 0.8109
 1500 chunks: 0.8081
Average ROUGE-L:
 500 chunks: 0.4952
 1000 chunks: 0.5024
 1500 chunks: 0.4926

EXPERIMENT 3
Average BERTScore:
 Top 3: 0.7843
 Top 5: 0.812
 Top 8: 0.8212
Average ROUGE-L:
 Top 3: 0.4445
 Top 5: 0.5036
 Top 8: 0.5213

